In [ ]:
'''
In LangGraph, Persistence means the ability to save and restore the state of a graph execution. It allows your workflow or agent to remember previous steps, continue from where it stopped, and maintain conversation history across multiple interactions.

Why Persistence is Important

Without persistence:

Every graph execution starts from scratch.
Conversation history is lost.
Long-running workflows cannot be resumed.

With persistence:

State is automatically saved after each node execution.
You can resume interrupted workflows.
Agents can remember previous conversations.
Human-in-the-loop workflows become possible.
'''

from langgraph.graph import StateGraph,START, END
from typing import TypedDict
from langchain_community.chat_models import ChatOllama
from langgraph.checkpoint.memory import InMemorySaver

In [3]:
llm  = ChatOllama(model="qwen3:8b")

C:\Users\Swati\AppData\Local\Temp\ipykernel_12580\2117186154.py:1: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  llm  = ChatOllama(model="qwen3:8b")


In [4]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str

In [8]:
def generate_joke(state: JokeState):
    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [9]:
def generate_explanation(state: JokeState):
    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [10]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [11]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': '**Joke:**  \n"Why did the pizza cross the road? To get to the other *slice*!" 🍕  \n\n**Bonus:**  \n*Why did the pizza get kicked out of the house? Because it was too cheesy—and still had toppings on its crust!* 😄',
 'explanation': '**Explanation of the Joke:**  \nThe joke "Why did the pizza cross the road? To get to the other *slice*!" is a playful pun that reimagines the classic "Why did the chicken cross the road?" setup. The humor hinges on the word **"slice"**, which has two meanings:  \n1. A **piece of pizza** (as in "a slice of pizza").  \n2. The **other side** of something (as in "the other side of the road").  \n\nBy replacing "the other side" with "the other slice," the joke cleverly ties the action (crossing the road) to the pizza\'s inherent feature (being divided into slices). It’s a light-hearted, wordplay-driven joke that relies on the listener’s familiarity with the classic setup and the double meaning of "slice."  \n\n---\n\n**Bonus Joke Exp

In [13]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': '**Joke:**  \n"Why did the pizza cross the road? To get to the other *slice*!" 🍕  \n\n**Bonus:**  \n*Why did the pizza get kicked out of the house? Because it was too cheesy—and still had toppings on its crust!* 😄', 'explanation': '**Explanation of the Joke:**  \nThe joke "Why did the pizza cross the road? To get to the other *slice*!" is a playful pun that reimagines the classic "Why did the chicken cross the road?" setup. The humor hinges on the word **"slice"**, which has two meanings:  \n1. A **piece of pizza** (as in "a slice of pizza").  \n2. The **other side** of something (as in "the other side of the road").  \n\nBy replacing "the other side" with "the other slice," the joke cleverly ties the action (crossing the road) to the pizza\'s inherent feature (being divided into slices). It’s a light-hearted, wordplay-driven joke that relies on the listener’s familiarity with the classic setup and the double meaning of "slice."  \n\n---\

In [14]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': '**Joke:**  \n"Why did the pizza cross the road? To get to the other *slice*!" 🍕  \n\n**Bonus:**  \n*Why did the pizza get kicked out of the house? Because it was too cheesy—and still had toppings on its crust!* 😄', 'explanation': '**Explanation of the Joke:**  \nThe joke "Why did the pizza cross the road? To get to the other *slice*!" is a playful pun that reimagines the classic "Why did the chicken cross the road?" setup. The humor hinges on the word **"slice"**, which has two meanings:  \n1. A **piece of pizza** (as in "a slice of pizza").  \n2. The **other side** of something (as in "the other side of the road").  \n\nBy replacing "the other side" with "the other slice," the joke cleverly ties the action (crossing the road) to the pizza\'s inherent feature (being divided into slices). It’s a light-hearted, wordplay-driven joke that relies on the listener’s familiarity with the classic setup and the double meaning of "slice."  \n\n---

In [12]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': '**Joke:**  \n"Why did the pasta go to the doctor? It was feeling a little *al dente*!"  \n\n*(A pun on "al dente" — the ideal texture of cooked pasta — and the phrase "feeling al dente," implying it was "firm" or "unemotional." Perfect for a noodle-noodle joke!)* 😄🍝',
 'explanation': 'The joke plays on the dual meaning of the Italian term **"al dente"**, which translates to "to the tooth" in English. In culinary contexts, "al dente" describes pasta that is cooked until it is firm to the bite—perfectly textured, not overcooked or mushy. \n\nThe humor arises from a pun: the phrase **"feeling al dente"** is a clever twist on the phrase **"feeling unemotional"** or **"feeling firm"**. The pasta is "feeling" (as in, in a state of being) "al dente," which humorously implies it’s too firm or unemotional, prompting it to visit the doctor. The joke hinges on the absurdity of a pasta needing medical attention for its texture, while also cleverly blending the culinary

In [15]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f169727-0f6d-6b98-8000-0ee2ff1323f0"}})

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f169727-0f6d-6b98-8000-0ee2ff1323f0'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-06-16T10:59:26.761052+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f169727-0f65-65cb-bfff-67b4817eaa38'}}, tasks=(PregelTask(id='3e2f1466-0786-69e9-ca98-064900a612bb', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': '**Joke:**  \n"Why did the pizza cross the road? To get to the other *slice*!" 🍕  \n\n**Bonus:**  \n*Why did the pizza get kicked out of the house? Because it was too cheesy—and still had toppings on its crust!* 😄'}),), interrupts=())

In [16]:
# execute that node again to see the state 
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f169727-0f6d-6b98-8000-0ee2ff1323f0"}})

{'topic': 'pizza',
 'joke': '**Joke:**  \n"Why did the pizza cross the road? To get to the other *slice*!" 🍕  \n\n*(A play on the classic "Why did the chicken cross the road?" joke, with a cheesy twist!)* 😄',
 'explanation': 'The joke plays on the classic "Why did the chicken cross the road?" punchline, which is typically answered with "To get to the other side!" Here, the humor comes from replacing "side" with **"slice"**, a word that has a double meaning:  \n1. **A part of a pizza** (since a pizza is divided into slices).  \n2. **The other side of the road** (as in the original joke).  \n\nBy substituting "side" with "slice," the joke creates a pun that ties the pizza\'s physical form (slices) to the act of crossing the road. The humor is cheesy and playful, relying on the absurdity of the scenario and the clever wordplay. The 🍕 emoji and 😄 reaction further emphasize the lighthearted, food-themed twist! 🥲'}

In [ ]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f169727-0f6d-6b98-8000-0ee2ff1323f0", "checkpoint_ns": ""}}, {'topic':'samosa'})

In [ ]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f169727-0f6d-6b98-8000-0ee2ff1323f0"}})